In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
sackmann = pd.read_csv("datos_sackmann.csv", encoding='latin1')
tennis_data = pd.read_csv("datos_tennis-data.csv", encoding='latin1')

C:\Users\User\AppData\Local\Temp\ipykernel_1860\2587999489.py:2: DtypeWarning: Columns (14,15,16,17,25,28) have mixed types. Specify dtype option on import or set low_memory=False.
  tennis_data = pd.read_csv("datos_tennis-data.csv", encoding='latin1')


In [3]:
sackmann.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101656 entries, 0 to 101655
Data columns (total 56 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   TOURNEY_ID          101656 non-null  object 
 1   TOURNEY_NAME        101656 non-null  object 
 2   SURFACE             101656 non-null  object 
 3   DRAW_SIZE           101656 non-null  int64  
 4   TOURNEY_LEVEL       101656 non-null  object 
 5   TOURNEY_DATE        101656 non-null  object 
 6   WINNER_ID           101656 non-null  int64  
 7   WINNER_SEED         45416 non-null   float64
 8   WINNER_NAME         101656 non-null  object 
 9   WINNER_HAND         101656 non-null  object 
 10  WINNER_HT           101529 non-null  float64
 11  WINNER_IOC          101656 non-null  object 
 12  WINNER_AGE          101656 non-null  float64
 13  LOSER_ID            101656 non-null  int64  
 14  LOSER_SEED          25095 non-null   float64
 15  LOSER_NAME          101656 non-nul

In [4]:
# Estandarizamos los apellidos
def surname_sackmann(name):
    # Divide el nombre por espacios y guiones
    divided_name = re.split(r'[\s\-]', str(name).strip())
    
    # Devuelve la última palabra del nombre (apellido) en minúsculas
    return divided_name[-1].lower()

def surname_tennis_data(name):
    # Elimina cualquier letra mayúscula seguida de un punto (abreviación nombre jugador)
    name_no_initials = re.sub(r'\s+[A-Za-z\.]+$', '', str(name)).strip()

    # Divide el nombre por espacios y guiones
    parts = re.split(r'[\s\-]', name_no_initials)
    
    # Devuelve la última palabra del nombre (apellido) en minúsculas
    return parts[-1].lower()

sackmann['W_LAST_NAME'] = sackmann['WINNER_NAME'].apply(surname_sackmann)
sackmann['L_LAST_NAME'] = sackmann['LOSER_NAME'].apply(surname_sackmann)

tennis_data['W_LAST_NAME'] = tennis_data['WINNER'].apply(surname_tennis_data)
tennis_data['L_LAST_NAME'] = tennis_data['LOSER'].apply(surname_tennis_data)

In [5]:
# Estandarizamos las fechas
sackmann['TOURNEY_DATE'] = pd.to_datetime(sackmann['TOURNEY_DATE'], errors='coerce')
tennis_data['DATE'] = pd.to_datetime(tennis_data['DATE'], errors='coerce')

In [6]:
tennis_data['W2'] = pd.to_numeric(tennis_data['W2'], errors='coerce')
tennis_data['L2'] = pd.to_numeric(tennis_data['L2'], errors='coerce')
tennis_data['W3'] = pd.to_numeric(tennis_data['W3'], errors='coerce')
tennis_data['L3'] = pd.to_numeric(tennis_data['L3'], errors='coerce')

In [7]:
df = pd.merge(
    # Datasets a mezclar
    sackmann, 
    tennis_data,
    # Si estas columnas son iguales en ambos datasets, las une
    on=['W_LAST_NAME', 'L_LAST_NAME', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5'],
    # Si una fila no tiene 'match', se queda
    how='left',
    # Si dos columnas tienen el mismo nombre, se añade un sujijo
    suffixes=('_sack', '_td')
)

In [8]:
df = df.drop(columns=['LOCATION','SURFACE_td','TOURNEY_LEVEL','ROUND_td','W_LAST_NAME','L_LAST_NAME','TOURNAMENT','DATE','BEST OF','WINNER',
                      'LOSER','WRANK','LRANK','WPTS', 'LPTS'])

In [9]:
df = df.rename(columns={'SURFACE_sack': 'SURFACE', 'ROUND_sack': 'ROUND'})

In [10]:
# Queremos cambiar la distribución Winner-Loser por Favorito-No_Favorito

# ¿Ha ganado el partido el jugador de peor ranking?
upset = df['WINNER_RANK'] > df['LOSER_RANK']
df['UPSET'] = upset.astype(int)

# Lista de columnas a intercambiar
winner_loser_tuples = [
    ('WINNER_ID','LOSER_ID'),('WINNER_SEED','LOSER_SEED'),('WINNER_NAME','LOSER_NAME'),('WINNER_HAND','LOSER_HAND'),('WINNER_HT','LOSER_HT'),
    ('WINNER_IOC','LOSER_IOC'),('WINNER_AGE','LOSER_AGE'),('W_ACE','L_ACE'),('W_DF','L_DF'),('W_SVPT','L_SVPT'),('W_1STIN','L_1STIN'),
    ('W_1STWON','L_1STWON'),('W_2NDWON','L_2NDWON'),('W_SVGMS','L_SVGMS'),('W_BPSAVED','L_BPSAVED'),('W_BPFACED','L_BPFACED'),
    ('WINNER_RANK','LOSER_RANK'),('WINNER_RANK_POINTS','LOSER_RANK_POINTS'),('W1','L1'),('W2','L2'),('W3','L3'),('W4','L4'),('W5','L5'),
    ('WSETS', 'LSETS'),('B365W', 'B365L'),('PSW', 'PSL')
]

# Intercambiamos Winner y Loser por Favorito y No Favorito
for winner_columns, loser_columns in winner_loser_tuples:
    # Nombre neutro para la nueva columna
    favourite = winner_columns.replace('WINNER_', '').replace('W_', '').replace('W', '') + '_FAVOURITE'
    underdog = loser_columns.replace('LOSER_', '').replace('L_', '').replace('L', '') + '_UNDERDOG'
    
    # Si upset: FAVOURITE es LOSER y el UNDERDOG es WINNER
    df[favourite] = np.where(upset, df[loser_columns], df[winner_columns])
    df[underdog] = np.where(upset, df[winner_columns], df[loser_columns])

# Eliminar columnas originales (winner - loser)
original_columns = [c for winner_loser_tuple in winner_loser_tuples for c in winner_loser_tuple]
df.drop(columns=original_columns, inplace=True)

In [11]:
df['TOURNEY_DATE'] = pd.to_datetime(df['TOURNEY_DATE'])

# Define el orden las rondas de los torneos
rounds = ['RR', 'BR', 'ER', 'R128', 'R64', 'R32', 'R16', 'QF', 'SF', 'F']

# Convierte la columna ROUND a un tipo categórico con el orden
df['ROUND'] = pd.Categorical(df['ROUND'], categories=rounds, ordered=True)

# Ordenamos usando los 3 niveles: fecha torneo, nombre torneo y ronda
df = df.sort_values(['TOURNEY_DATE', 'TOURNEY_NAME', 'ROUND']).reset_index(drop=True)

In [12]:
df.shape

(102817, 65)

In [13]:
print(f"Dimensiones antes de limpiar: {df.shape}")

# Textos que indican anomalía en los datasets
anomalous_strings = ['RET', 'W/O', 'DEF', 'Default', 'Walkover']

# Filtro para la columna SCORE
mask_score = ~df['SCORE'].astype(str).str.upper().str.contains('|'.join(anomalous_strings), na=False)

# Filtro para la columna COMMENT (si es NaN, asumimos que se completó)
mask_comment = ~df['COMMENT'].fillna('Completed').str.contains('Retired|Walkover|W/O|DEF', case=False, na=False)

# Filtro de seguridad: Partidos que duraron menos de 15 minutos suelen ser retiradas no registradas bien
df = df[(df['MINUTES'] > 15) | (df['MINUTES'].isna())]

# Aplicar filtros
df = df[mask_score & mask_comment].reset_index(drop=True)
print(f"Dimensiones tras limpiar: {df.shape}")

Dimensiones antes de limpiar: (102817, 65)
Dimensiones tras limpiar: (99517, 65)


C:\Users\User\AppData\Local\Temp\ipykernel_1860\3489150046.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[mask_score & mask_comment].reset_index(drop=True)


In [14]:
df = df.sort_values(['TOURNEY_DATE', 'TOURNEY_NAME', 'ROUND']).reset_index(drop=True)

In [15]:
df

,TOURNEY_ID,TOURNEY_NAME,SURFACE,DRAW_SIZE,TOURNEY_DATE,SCORE,BEST_OF,ROUND,MINUTES,SERIES,...,4_FAVOURITE,4_UNDERDOG,5_FAVOURITE,5_UNDERDOG,SETS_FAVOURITE,SETS_UNDERDOG,B365_FAVOURITE,B365_UNDERDOG,PS_FAVOURITE,PS_UNDERDOG
0,1991-339,Adelaide,Hard,32,1990-12-31,6-4 3-6 7-6(2),3,R32,130.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1991-339,Adelaide,Hard,32,1990-12-31,6-0 6-4,3,R32,71.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1991-339,Adelaide,Hard,32,1990-12-31,7-6(2) 6-1,3,R32,85.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1991-339,Adelaide,Hard,32,1990-12-31,7-5 6-3,3,R32,90.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1991-339,Adelaide,Hard,32,1990-12-31,6-2 6-3,3,R32,88.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99512,2026-0414,Hamburg,Clay,32,2026-05-17,6-4 6-7(10) 6-2,3,QF,163.0,ATP 500,...,NaN,NaN,NaN,NaN,1.0,2.0,1.4,3.0,NaN,NaN
99513,2026-0414,Hamburg,Clay,32,2026-05-17,6-2 7-5,3,QF,98.0,ATP 500,...,NaN,NaN,NaN,NaN,2.0,0.0,1.4,3.0,NaN,NaN
99514,2026-0414,Hamburg,Clay,32,2026-05-17,2-6 6-3 6-3,3,SF,137.0,ATP 500,...,NaN,NaN,NaN,NaN,1.0,2.0,1.57,2.38,NaN,NaN
99515,2026-0414,Hamburg,Clay,32,2026-05-17,6-1 6-4,3,SF,64.0,ATP 500,...,NaN,NaN,NaN,NaN,2.0,0.0,1.29,3.75,NaN,NaN


In [16]:
df.to_csv('datos_conjuntos.csv', index=False, encoding='latin1')